#### How to handle Bad / Corrupt records while reading json / csv files?
- In PySpark, you can handle **corrupted records** such as those with **data type mismatches or incorrect delimiters** using **three primary read modes** and specific configurations.

**Core Read Modes:**
- When reading files **(CSV, JSON)**, you can specify how Spark should react to **malformed data** using the **.option("mode", "MODE_NAME")** setting.

| Option                     | Purpose                                        |
| -------------------------- | ---------------------------------------------- |
| `PERMISSIVE` (default)     | Spark **loads the data** but inserts **NULL** for fields it **cannot parse**. Stores bad records in **_corrupt_record** |
| `DROPMALFORMED`            | Spark **ignores and drops** any record that contains **malformed data**, returning **only valid rows**.  |
| `FAILFAST`                 | **Fails immediately on bad JSON**. Spark immediately stops execution and **throws an exception** as soon as it encounters a **single corrupt record**.  |

**Content:**
- **1) PERMISSIVE**
  - `How to Isolate and Quarantine Corrupt Rows using PERMISSIVE?`
  - `When DOES columnNameOfCorruptRecord matter?`
  - `What if without bad_record in schema?`
  - `What if schema has bad_record but option uses different name?`
  - `What if both Schema & option has same names?`
  - `How to Extract Only Bad Records?`
  - `How to Extract Only Valid Records?`
  - `singleLine List`
- **2) Drop Corrupt Records using DROPMALFORMED**
  - `When to Use DROPMALFORMED?`
  - `Difference between PERMISSIVE & DROPMALFORMED?`
- **3) Fail on Bad Records FAILFAST**
- **4) How to write corrupted JSON objects into an external directory?**

#### Implementation Guide

**1) Strategy A => PERMISSIVE**

**How to Isolate and Quarantine Corrupt Rows using PERMISSIVE?**.
- Useful when **one bad JSON block** can **break the entire file**.
- The best practice for **production data pipelines** is to keep **good data moving** while **storing malformed records separately**.
- To do this, you must **explicitly define a schema** and **include a string column** designated for **bad records**.

      from pyspark.sql.types import StructType, StructField, StringType, IntegerType
      import pyspark.sql.functions as F

      1) Define explicit schema including the special corrupt record tracking column
      schema = StructType([
          StructField("id", IntegerType(), True),
          StructField("name", StringType(), True),
          StructField("age", IntegerType(), True),
          StructField("_corrupt_record", StringType(), True) # Must match the option string
      ])

      2) Read file with permissive mode configured to populate our custom column
      # Read JSON file
      df = (spark.read
          .format("json")
          .option(multiline, True) -> multiline json
          .schema(schema)
          .option("mode", "PERMISSIVE")
          .option("columnNameOfCorruptRecord", "_corrupt_record")
          .load("path/to/data.json"))

      # Read csv file
      df = (spark.read
          .format("csv")
          .option("header", "true")
          .schema(schema)
          .option("mode", "PERMISSIVE")
          .option("columnNameOfCorruptRecord", "_corrupt_record")
          .load("path/to/data.csv"))

      3) Separate clean records from corrupt ones
      clean_df = df.filter(F.col("_corrupt_record").isNull()).drop("_corrupt_record")
      corrupt_df = df.filter(F.col("_corrupt_record").isNotNull()).select("_corrupt_record")

      4) Process clean data and quarantine bad data
      clean_df.write.format("parquet").save("path/to/clean_data")
      corrupt_df.write.format("text").save("path/to/corrupt_quarantine")

**a) Reading with / without PERMISSIVE**
- `PERMISSIVE` is the **DEFAULT** read mode in Spark JSON reader

      # Method 01
      spark.read.json("events_singleline.json")

      # Method 02
      spark.read.option("mode", "PERMISSIVE").json("events_singleline.json")

In [0]:
# we don't get column of "_corrupt_record" for clean data
df_default = spark.read \
    .json("/Volumes/azureadb/pyspark/training/json/read_json/SingleLine/singleline.json")

display(df_default)

Country2,description,input_timestamp,last_update_timestamp,source,user
IND,bravia,1124256609,1524256609,catalog,Hari
US,sony,1224256609,1424256609,SAP,Rajesh
CANADA,bse,1324256609,1524256609,ADLS,Lokesh
US,exchange,1424256609,1724256609,Blob,Sharath
SWEDEN,Stock,1524256609,1664256609,SQL,Sheetal
UK,azure,1624256609,1874256609,datawarehouse,Raj
Norway,ADF,1779256609,188256609,oracle,Synapse


In [0]:
df_wo_perm = spark.read \
    .json("/Volumes/azureadb/pyspark/training/json/read_json_permissive/events_singleline.json")

display(df_wo_perm)
df_wo_perm.printSchema()

_corrupt_record,age,id,name
null,30,1,John
null,28,2,Suresh
null,25,3,Anita
null,34,4,John
null,31,5,Sandya
null,24,6,Ashish
"{""id"":7,""name"":""Ravi"",""age"": }",null,null,null
null,ABC,8,Rakesh
null,35,9,Meena


root
 |-- _corrupt_record: string (nullable = true)
 |-- age: string (nullable = true)
 |-- id: long (nullable = true)
 |-- name: string (nullable = true)



In [0]:
df_w_perm = spark.read \
    .option("mode", "PERMISSIVE") \
    .json("/Volumes/azureadb/pyspark/training/json/read_json_permissive/events_singleline.json")

display(df_w_perm)

_corrupt_record,age,id,name
null,30,1,John
null,28,2,Suresh
null,25,3,Anita
null,34,4,John
null,31,5,Sandya
null,24,6,Ashish
"{""id"":7,""name"":""Ravi"",""age"": }",null,null,null
null,ABC,8,Rakesh
null,35,9,Meena


In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

In [0]:
# _corrupt_record is not part of the schema
schema_wo_recrd = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("age", IntegerType(), True)
])

df_w_perm = spark.read \
    .schema(schema_wo_recrd) \
    .option("mode", "PERMISSIVE") \
    .json("/Volumes/azureadb/pyspark/training/json/read_json_permissive/events_singleline.json")

display(df_w_perm)
df_w_perm.printSchema()

id,name,age
1,John,30
2,Suresh,28
3,Anita,25
4,John,34
5,Sandya,31
6,Ashish,24
null,null,null
8,Rakesh,null
9,Meena,35


root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- age: integer (nullable = true)



In [0]:
schema_w_recrd = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("_corrupt_record", StringType(), True)
])

# .option("columnNameOfCorruptRecord", "_corrupt_record") \
df_w_perm_recrd = spark.read \
    .schema(schema_w_recrd) \
    .option("mode", "PERMISSIVE") \
    .json("/Volumes/azureadb/pyspark/training/json/read_json_permissive/events_singleline.json")

display(df_w_perm_recrd)

id,name,age,_corrupt_record
1,John,30,null
2,Suresh,28,null
3,Anita,25,null
4,John,34,null
5,Sandya,31,null
6,Ashish,24,null
null,null,null,"{""id"":7,""name"":""Ravi"",""age"": }"
8,Rakesh,null,"{""id"":8,""name"":""Rakesh"",""age"":""ABC"" }"
9,Meena,35,null


- **Bad row safely captured**.
- **File does NOT fail**.

**b) When DOES `columnNameOfCorruptRecord` matter?**
- It **only matters** when you **change the column name**.

In [0]:
# infer schema: data type of "Age" detecting as "string" due to record of "ABC"
df_wo_perm_recrd = spark.read \
    .option("mode", "PERMISSIVE") \
    .option("columnNameOfCorruptRecord", "bad_record") \
    .json("/Volumes/azureadb/pyspark/training/json/read_json_permissive/events_singleline.json")

display(df_wo_perm_recrd)

age,bad_record,id,name
30,null,1,John
28,null,2,Suresh
25,null,3,Anita
34,null,4,John
31,null,5,Sandya
24,null,6,Ashish
null,"{""id"":7,""name"":""Ravi"",""age"": }",null,null
ABC,null,8,Rakesh
35,null,9,Meena


##### c) What if without `bad_record` in schema?

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

# bad_record is not part of the schema
schema_wo_recrd = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("age", IntegerType(), True)
])

df_wo_perm_recrd = spark.read \
    .schema(schema_wo_recrd) \
    .option("mode", "PERMISSIVE") \
    .option("columnNameOfCorruptRecord", "bad_record") \
    .json("/Volumes/azureadb/pyspark/training/json/read_json_permissive/events_singleline.json")

display(df_wo_perm_recrd)

id,name,age
1,John,30
2,Suresh,28
3,Anita,25
4,John,34
5,Sandya,31
6,Ashish,24
null,null,null
8,Rakesh,null
9,Meena,35


In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

# bad_record is part of the schema
schema_wo_diffbadrecrd = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("bad_record", StringType(), True)
])

df_wo_perm_diffbadrecrd_bad = spark.read \
    .schema(schema_wo_diffbadrecrd) \
    .option("mode", "PERMISSIVE") \
    .option("columnNameOfCorruptRecord", "bad_record") \
    .json("/Volumes/azureadb/pyspark/training/json/read_json_permissive/events_singleline.json")

display(df_wo_perm_diffbadrecrd_bad)
df_wo_perm_diffbadrecrd_bad.printSchema()

id,name,age,bad_record
1,John,30,null
2,Suresh,28,null
3,Anita,25,null
4,John,34,null
5,Sandya,31,null
6,Ashish,24,null
null,null,null,"{""id"":7,""name"":""Ravi"",""age"": }"
8,Rakesh,null,"{""id"":8,""name"":""Rakesh"",""age"":""ABC"" }"
9,Meena,35,null


root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- bad_record: string (nullable = true)



##### d) What if schema has `bad_record` but option uses `different name`?

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

# bad_record is part of the schema
schema_wo_diffbadrecrd = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("bad_record", StringType(), True)
])

df_wo_perm_diffbadrecrd = spark.read \
    .schema(schema_wo_diffbadrecrd) \
    .option("mode", "PERMISSIVE") \
    .option("columnNameOfCorruptRecord", "_corrupt_record") \
    .json("/Volumes/azureadb/pyspark/training/json/read_json_permissive/events_singleline.json")

display(df_wo_perm_diffbadrecrd)
df_wo_perm_diffbadrecrd.printSchema()

id,name,age,bad_record
1,John,30,null
2,Suresh,28,null
3,Anita,25,null
4,John,34,null
5,Sandya,31,null
6,Ashish,24,null
null,null,null,null
8,Rakesh,null,null
9,Meena,35,null


root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- bad_record: string (nullable = true)



##### e) What if `both Schema & option` has same names?

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

# bad_record is part of the schema
schema_wo_badrecrd = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("bad_record", StringType(), True)
])

df_wo_perm_badrecrd = spark.read \
    .schema(schema_wo_badrecrd) \
    .option("mode", "PERMISSIVE") \
    .option("columnNameOfCorruptRecord", "bad_record") \
    .json("/Volumes/azureadb/pyspark/training/json/read_json_permissive/events_singleline.json")

display(df_wo_perm_badrecrd)

id,name,age,bad_record
1,John,30,null
2,Suresh,28,null
3,Anita,25,null
4,John,34,null
5,Sandya,31,null
6,Ashish,24,null
null,null,null,"{""id"":7,""name"":""Ravi"",""age"": }"
8,Rakesh,null,"{""id"":8,""name"":""Rakesh"",""age"":""ABC"" }"
9,Meena,35,null


**f) How to Extract Only Bad Records?**

In [0]:
bad_df = df_w_perm_recrd.filter("_corrupt_record is not null")
display(bad_df)

id,name,age,_corrupt_record
null,null,null,"{""id"":7,""name"":""Ravi"",""age"": }"
8,Rakesh,null,"{""id"":8,""name"":""Rakesh"",""age"":""ABC"" }"


**g) How to Extract Only Valid Records?**

In [0]:
clean_df = df_w_perm_recrd.filter("_corrupt_record is null") \
    .drop("_corrupt_record")

display(clean_df)

id,name,age
1,John,30
2,Suresh,28
3,Anita,25
4,John,34
5,Sandya,31
6,Ashish,24
8,Meena,35


**h) singleLine List json**

In [0]:
df_mlt_default = spark.read \
                      .option("multiLine","true") \
                      .option("mode","PERMISSIVE") \
                      .json("/Volumes/azureadb/pyspark/training/json/read_json_permissive/events_multiline.json")

display(df_mlt_default)

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-6628883060045275>, line 6
      1 df_mlt_default = spark.read \
      2                       .option("multiLine","true") \
      3                       .option("mode","PERMISSIVE") \
      4                       .json("/Volumes/azureadb/pyspark/training/json/read_json_permissive/events_multiline.json")
----> 6 display(df_mlt_default)

File /databricks/python_shell/lib/dbruntime/display.py:133, in Display.display(self, input, *args, **kwargs)
    131     pass
    132 elif self._cf_helper is not None and isinstance(input, ConnectDataFrame):
--> 133     self.display_connect_table(input, **kwargs)
    134 elif isinstance(input, ConnectDataFrame):
    135     if input.isStreaming:

File /databricks/python_shell/lib/dbruntime/display.py:97, in Display.display_connect_table(self, df, **kwargs)
     94     self.cf_helper.displa

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

schema_w_recrd = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("_corrupt_record", StringType(), True)
])

df_mlt = spark.read \
              .schema(schema_w_recrd) \
              .option("multiLine","true") \
              .option("mode","PERMISSIVE") \
              .json("/Volumes/azureadb/pyspark/training/json/read_json_permissive/events_multiline.json")

display(df_mlt)
df_mlt.printSchema()

id,name,age,_corrupt_record
1,A,20,null
2,B,25,null
3,C,30,null
4,John,30,null
5,Suresh,28,null
6,Anita,25,null
7,John,34,null
8,Sandya,31,null
9,Ashish,24,null
null,null,null,"[ {""id"":1,""name"":""A"",""age"":20}, {""id"":2,""name"":""B"",""age"":25}, {""id"":3,""name"":""C"",""age"":30}, {""id"":4,""name"":""John"",""age"":30}, {""id"":5,""name"":""Suresh"",""age"":28}, {""id"":6,""name"":""Anita"",""age"":25}, {""id"":7,""name"":""John"",""age"":34}, {""id"":8,""name"":""Sandya"",""age"":31}, {""id"":9,""name"":""Ashish"",""age"":24}, {""id"":10,""name"":""Ravi"",""age"": }, {""id"":11,""name"":""Meena"",""age"":35}, {""id"":12,""name"":""Rudra"",""age"":30}, {""id"":13,""name"":""Ragini"",""age"":32} ]"


root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- _corrupt_record: string (nullable = true)



In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

schema_w_recrd_mlt = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("bad_record", StringType(), True)
])

df_mlt_bad = spark.read \
              .schema(schema_w_recrd_mlt) \
              .option("multiLine","true") \
              .option("mode","PERMISSIVE") \
              .option("columnNameOfCorruptRecord", "bad_record") \
              .json("/Volumes/azureadb/pyspark/training/json/read_json_permissive/events_multiline.json")

display(df_mlt_bad)
df_mlt_bad.printSchema()

id,name,age,bad_record
1,A,20,null
2,B,25,null
3,C,30,null
4,John,30,null
5,Suresh,28,null
6,Anita,25,null
7,John,34,null
8,Sandya,31,null
9,Ashish,24,null
null,null,null,"[ {""id"":1,""name"":""A"",""age"":20}, {""id"":2,""name"":""B"",""age"":25}, {""id"":3,""name"":""C"",""age"":30}, {""id"":4,""name"":""John"",""age"":30}, {""id"":5,""name"":""Suresh"",""age"":28}, {""id"":6,""name"":""Anita"",""age"":25}, {""id"":7,""name"":""John"",""age"":34}, {""id"":8,""name"":""Sandya"",""age"":31}, {""id"":9,""name"":""Ashish"",""age"":24}, {""id"":10,""name"":""Ravi"",""age"": }, {""id"":11,""name"":""Meena"",""age"":35}, {""id"":12,""name"":""Rudra"",""age"":30}, {""id"":13,""name"":""Ragini"",""age"":32} ]"


root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- bad_record: string (nullable = true)



**Recommendation**

- Always use **PERMISSIVE** in **Bronze / Raw** layers.
- Log **_corrupt_record** for `debugging`.
- `Clean data` in **Silver layer**.

##### 2) Strategy B => Drop Corrupt Records

**Ignore Malformed Rows (DROPMALFORMED)**
- If you prefer to **bypass corrupt data** entirely **without writing quarantine log files**, use the **DROPMALFORMED** mode.

     df_cleaned = (spark.read
                 .format("json")
                 .schema(schema)
                 .option("mode", "DROPMALFORMED")
                 .load("path/to/data.json"))

**When to Use DROPMALFORMED?**

- After data validation.
- Clean analytics datasets.
- When `bad data is not required`.
- `Silver / Gold layers`.

**Important Warning**

- `DROPMALFORMED` **drops rows silently**.
- You `will NOT know` what was **removed**.

In [0]:
schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("age", IntegerType(), True)
])

df_drop = spark.read \
               .schema(schema) \
               .option("mode", "DROPMALFORMED") \
               .json("/Volumes/azureadb/pyspark/training/json/read_json_permissive/events_singleline.json")

display(df_drop)

id,name,age
1,John,30
2,Suresh,28
3,Anita,25
4,John,34
5,Sandya,31
6,Ashish,24
9,Meena,35


In [0]:
df_mlt_drop = spark.read \
                   .schema(schema) \
                   .option("multiLine", "true") \
                   .option("mode", "DROPMALFORMED") \
                   .json("/Volumes/azureadb/pyspark/training/json/read_json_permissive/events_multiline.json")
display(df_mlt_drop)

id,name,age
1,A,20
2,B,25
3,C,30
4,John,30
5,Suresh,28
6,Anita,25
7,John,34
8,Sandya,31
9,Ashish,24


❌ Bad record is silently dropped

✔ Job succeeds

✔ Only clean data remains

**Key Difference PERMISSIVE Vs DROPMALFORMED**

| Feature           | PERMISSIVE   | DROPMALFORMED   |
| ----------------- | ------------ | --------------- |
| Bad rows kept     | ✅ Yes        | ❌ No            |
| `_corrupt_record` | ✅ Available  | ❌ Not available |
| Data audit        | ✅ Possible   | ❌ Lost          |
| Best for          | Bronze / Raw | Silver / Clean  |

##### 3) Strategy C => FAILFAST

**Fail on Bad Records**

- If data integrity is vital and **any error** should **immediately stop your data pipeline**, run the script with **FAILFAST**.
- Wrap it in a Python **try-except** block to gracefully catch the issue.

     try:
         df = (spark.read
             .format("csv")
             .schema(schema)
             .option("mode", "FAILFAST")
             .load("path/to/data.csv"))
         df.count() # Force action to trigger record parsing
     except Exception as e:
         print(f"Data ingestion stopped due to corruption: {e}")

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("age", IntegerType(), True)
])

df_failfast_single = spark.read \
                          .schema(schema) \
                          .option("mode", "FAILFAST") \
                          .json("/Volumes/azureadb/pyspark/training/json/read_json_permissive/events_singleline.json")

display(df_failfast_single)

---------------------------------------------------------------------------
SparkException                            Traceback (most recent call last)
File <command-7995494653691701>, line 14
      3 schema = StructType([
      4     StructField("id", IntegerType(), True),
      5     StructField("name", StringType(), True),
      6     StructField("age", IntegerType(), True)
      7 ])
      9 df_failfast_single = spark.read \
     10                           .schema(schema) \
     11                           .option("mode", "FAILFAST") \
     12                           .json("/Volumes/azureadb/pyspark/training/json/read_json_permissive/events_singleline.json")
---> 14 display(df_failfast_single)

File /databricks/python_shell/lib/dbruntime/display.py:133, in Display.display(self, input, *args, **kwargs)
    131     pass
    132 elif self._cf_helper is not None and isinstance(input, ConnectDataFrame):
--> 133     self.display_connect_table(input, **kwargs)
    134 elif isinstance

- Spark immediately `throws an exception`.
- `No DataFrame` is created.
- `Job fails` as soon as the `bad record` is hit.
- `No table output`.
- The `job never reaches` **display(df_failfast)**.

In [0]:
df_failfast_mlt = spark.read \
                       .schema(schema) \
                       .option("multiLine", "true") \
                       .option("mode", "FAILFAST") \
                       .json("/Volumes/azureadb/pyspark/training/json/read_json_permissive/events_multiline.json")
display(df_failfast_mlt)

---------------------------------------------------------------------------
SparkException                            Traceback (most recent call last)
File <command-4885668766766076>, line 6
      1 df_failfast_mlt = spark.read \
      2                        .schema(schema) \
      3                        .option("multiLine", "true") \
      4                        .option("mode", "FAILFAST") \
      5                        .json("/Volumes/azureadb/pyspark/training/json/read_json_permissive/events_multiline.json")
----> 6 display(df_failfast_mlt)

File /databricks/python_shell/lib/dbruntime/display.py:133, in Display.display(self, input, *args, **kwargs)
    131     pass
    132 elif self._cf_helper is not None and isinstance(input, ConnectDataFrame):
--> 133     self.display_connect_table(input, **kwargs)
    134 elif isinstance(input, ConnectDataFrame):
    135     if input.isStreaming:

File /databricks/python_shell/lib/dbruntime/display.py:97, in Display.display_connect_table

In [0]:
try:
    df = (spark.read
        .schema(schema)
        .option("mode", "FAILFAST")
        .load("/Volumes/azureadb/pyspark/training/json/read_json_permissive/events_singleline.json"))
    df.count() # Force action to trigger record parsing
except Exception as e:
    print(f"Data ingestion stopped due to corruption: {e}")

Data ingestion stopped due to corruption: [DELTA_MISSING_TRANSACTION_LOG] Incompatible format detected.

You are trying to read from `/Volumes/azureadb/pyspark/training/json/read_json_permissive/events_singleline.json` using Delta, but there is no
transaction log present. Check the upstream job to make sure that it is writing
using format("delta") and that you are trying to read from the table base path.

To learn more about Delta, see https://docs.databricks.com/delta/index.html


JVM stacktrace:
com.databricks.sql.transaction.tahoe.DeltaAnalysisException
	at com.databricks.sql.transaction.tahoe.DeltaErrorsEdge.useDeltaOnOtherFormatPathException(DeltaErrorsEdge.scala:720)
	at com.databricks.sql.transaction.tahoe.DeltaErrorsEdge.useDeltaOnOtherFormatPathException$(DeltaErrorsEdge.scala:718)
	at com.databricks.sql.transaction.tahoe.DeltaErrors$.useDeltaOnOtherFormatPathException(DeltaErrors.scala:4417)
	at com.databricks.sql.transaction.tahoe.DeltaValidation$.validateDeltaRead(DeltaVali

In [0]:
schema_w_recrd_mlt = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("_corrupt_record", StringType(), True)
])

try:
    df1 = (spark.read
           .schema(schema_w_recrd_mlt)
        .option("mode", "PERMISSIVE")
        .json("/Volumes/azureadb/pyspark/training/json/read_json_permissive/events_singleline.json"))
    df1.display() # Force action to trigger record parsing
except Exception as e:
    print(f"Data ingestion stopped due to corruption: {e}")

id,name,age,_corrupt_record
1,John,30,null
2,Suresh,28,null
3,Anita,25,null
4,John,34,null
5,Sandya,31,null
6,Ashish,24,null
null,null,null,"{""id"":7,""name"":""Ravi"",""age"": }"
8,Rakesh,null,"{""id"":8,""name"":""Rakesh"",""age"":""ABC"" }"
9,Meena,35,null


In [0]:
try:
    df1 = (spark.read
        .schema(schema)
        .option("mode", "DROPMALFORMED")
        .json("/Volumes/azureadb/pyspark/training/json/read_json_permissive/events_singleline.json"))
    df1.display() # Force action to trigger record parsing
except Exception as e:
    print(f"Data ingestion stopped due to corruption: {e}")

id,name,age
1,John,30
2,Suresh,28
3,Anita,25
4,John,34
5,Sandya,31
6,Ashish,24
9,Meena,35


##### 4) Alternative: Routing to badRecordsPath
- Instead of **filtering** your DataFrame **manually** with PySpark options, you can delegate the separation entirely to Spark using the global **badRecordsPath** option.
- This approach writes **corrupted JSON objects** into an **external directory** background-style while your script loads only the clean records

**Your source file contains two problematic records:**

     {"id":7,"name":"Ravi","age": }
     {"id":8,"name":"Rakesh","age":"ABC" }

- `Record 7 is corrupt because salary should be an integer but it is empty.`
- `Record 8 is corrupt because salary should be an integer but passed as string.`

In [0]:
df_path = (spark.read
    .schema(schema_w_recrd_mlt)
    .option("badRecordsPath", "/Volumes/azureadb/pyspark/training/json/read_json_permissive/bad_records")
    .json("/Volumes/azureadb/pyspark/training/json/read_json_permissive/events_singleline.json"))

display(df_path)

id,name,age,_corrupt_record
1,John,30,null
2,Suresh,28,null
3,Anita,25,null
4,John,34,null
5,Sandya,31,null
6,Ashish,24,null
9,Meena,35,null


**Check What Spark Wrote to badRecordsPath**

In [0]:
display(
    dbutils.fs.ls(
        "/Volumes/azureadb/pyspark/training/json/read_json_permissive/bad_records"
    )
)

path,name,size,modificationTime
dbfs:/Volumes/azureadb/pyspark/training/json/read_json_permissive/bad_records/20260605T151336/,20260605T151336/,0,1780684492924
dbfs:/Volumes/azureadb/pyspark/training/json/read_json_permissive/bad_records/20260605T183444/,20260605T183444/,0,1780684492924


**Read the Corrupt Records**

In [0]:
bad_df = spark.read.json("/Volumes/azureadb/pyspark/training/json/read_json_permissive/bad_records/20260605T151336/bad_records/part-00000-ba7c3483-b4e9-4f98-82e3-a0b8f3b5a9c2")
display(bad_df)

reason,record
"com.fasterxml.jackson.core.JsonParseException: Unexpected character ('}' (code 125)): expected a value at [Source: REDACTED (`StreamReadFeature.INCLUDE_SOURCE_IN_LOCATION` disabled); line: 1, column: 30]","{""id"":7,""name"":""Ravi"",""age"": }"
"org.apache.spark.SparkRuntimeException: [CANNOT_PARSE_JSON_FIELD] Cannot parse the field name 'age' and the value ABC of the JSON token type VALUE_STRING to target Spark data type ""INT"". SQLSTATE: 2203G","{""id"":8,""name"":""Rakesh"",""age"":""ABC"" }"
